# 03 Machine Learning Modeling
This notebook was programmatically generated to demonstrate key project steps.


## 1. Import Packages & Load Preprocessed Data
We prepare scikit-learn preprocessing pipelines, run SMOTE to address class imbalance, and fit models.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
import shap
import matplotlib.pyplot as plt


## 2. Load Processed Dataset & Add Features


In [ ]:
df = pd.read_csv('../data/processed/hr_clean.csv')
joblevel_avg = df.groupby('JobLevel')['MonthlyIncome'].transform('mean')
df['income_ratio_to_joblevel_avg'] = df['MonthlyIncome'] / (joblevel_avg + 1e-5)
df['role_tenure_ratio'] = df['YearsInCurrentRole'] / (df['YearsAtCompany'] + 1)

X = df.drop(columns=['Attrition', 'EmployeeNumber'])
y = df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


## 3. Apply Column Preprocessing Pipeline


In [ ]:
cat_cols = ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'MaritalStatus', 'OverTime']
num_cols = [c for c in X.columns if c not in cat_cols]

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
])

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)


## 4. Run SMOTE for Class Imbalance


In [ ]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_prep, y_train)
print(f'Original Class Ratio: {np.bincount(y_train)}')
print(f'Resampled Class Ratio: {np.bincount(y_train_res)}')


## 5. Fit & Evaluate Baseline vs. XGBoost


In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_res, y_train_res)
lr_preds = lr.predict(X_test_prep)
lr_probs = lr.predict_proba(X_test_prep)[:, 1]
print('--- Baseline Logistic Regression ---')
print(classification_report(y_test, lr_preds))
print(f'ROC-AUC: {roc_auc_score(y_test, lr_probs):.4f}')

xgb = XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.08, eval_metric='logloss', random_state=42)
xgb.fit(X_train_res, y_train_res)
xgb_preds = xgb.predict(X_test_prep)
xgb_probs = xgb.predict_proba(X_test_prep)[:, 1]
print('\n--- XGBoost Classifier ---')
print(classification_report(y_test, xgb_preds))
print(f'ROC-AUC: {roc_auc_score(y_test, xgb_probs):.4f}')


## 6. Model Explainability with SHAP
Let's see what drivers are contributing to model predictions.


In [ ]:
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test_prep)
feature_names = [name.split('__')[1] for name in preprocessor.get_feature_names_out()]

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_prep, feature_names=feature_names, show=True)
